# 04 - Evaluate Continuous ISL Model & Run Sentence Inference

Loads the trained model checkpoint (`best.pt`), computes evaluation metrics (Word Error Rate & Exact Match), and executes the end-to-end continuous ISL inference pipeline:

$$\text{Continuous ISL Video / Keypoints} \longrightarrow \text{ST-GCN + Transformer} \longrightarrow \text{CTC Logits} \longrightarrow \text{English Word Tokens} \longrightarrow \text{Formatted Sentence}$$

> **Dataset Note:** The model predicts English word-token sequences derived from sentence transcript targets. These should not be interpreted as manually annotated ISL glosses.

In [ ]:
# 1. Import evaluation & inference modules
import os, sys
from pathlib import Path
import numpy as np

WORKING_DIR = Path("/kaggle/working/sentence-model-training") if os.path.exists("/kaggle") else Path("..")
sys.path.insert(0, str(WORKING_DIR.resolve()))

from src.evaluate import evaluate_model
from src.dataset import SentenceISLDataset
from src.inference import SentenceInferencePipeline
from src.vocabulary import GlossVocabulary, WordVocabulary
from src.formatter import format_gloss_to_sentence
print("Evaluation modules loaded successfully.")

In [ ]:
# 2. Demonstrate Word-Tokens to Formatted Sentence Conversion
sample_decoded_tokens = ["good", "morning"]
formatted_sentence = format_gloss_to_sentence(sample_decoded_tokens)
print(f"Raw Decoded Tokens: {sample_decoded_tokens}")
print(f"Formatted Sentence: '{formatted_sentence}'")

sample_phrase_2 = ["how", "are", "you"]
formatted_2 = format_gloss_to_sentence(sample_phrase_2)
print(f"Raw Decoded Tokens: {sample_phrase_2}")
print(f"Formatted Sentence: '{formatted_2}'")

In [ ]:
# 3. Run Evaluation on Validation Set (if checkpoint exists)
CHECKPOINT_PATH = Path("/kaggle/working/checkpoints/best.pt") if os.path.exists("/kaggle") else Path("../checkpoints/best.pt")

if CHECKPOINT_PATH.exists():
    dataset = SentenceISLDataset(debug_mode=True, debug_num_samples=16)
    results = evaluate_model(checkpoint_path=str(CHECKPOINT_PATH), dataset=dataset)
    print(f"Evaluation Results: {results}")
else:
    print(f"Checkpoint not found at {CHECKPOINT_PATH}. Run training (03_train_sentence_model.ipynb) first.")

In [ ]:
# 4. Test End-to-End Keypoint-to-Sentence Inference Pipeline
if CHECKPOINT_PATH.exists():
    pipeline = SentenceInferencePipeline(checkpoint_path=CHECKPOINT_PATH)
    
    # Dummy test keypoint sequence: T=32 frames, 543 landmarks, 3 coordinates (x, y, z)
    dummy_keypoints = np.random.randn(32, 543, 3).astype(np.float32)
    
    prediction = pipeline.predict_keypoints(dummy_keypoints)
    print("--- End-to-End Prediction Demo ---")
    print(f"Predicted English Tokens: {prediction.get('tokens', prediction.get('glosses'))}")
    print(f"Formatted Sentence:       '{prediction['sentence']}'")
    print(f"Processed Frames (T):     {prediction['frame_count']}")